# 02 — Data Preprocessing

| Step | What happens |
|---|---|
| 1 | Load raw articles |
| 2 | Inspect text before cleaning |
| 3 | Run 8-step clean_text() pipeline |
| 4 | Before/after comparison |
| 5 | Token count analysis |
| 6 | Vocabulary statistics |

> **Next step:** `03_EDA_Sentiment_Overview.ipynb`

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.config import NEWS_RAW_FILE

raw_df = pd.read_csv(NEWS_RAW_FILE)
print(f'Raw articles loaded: {len(raw_df)}')
raw_df[['title', 'text', 'source', 'keyword']].head(3)

## 1. Inspect Raw Text

In [ ]:
for i, row in raw_df.head(3).iterrows():
    print(f'[{i}] TITLE: {row["title"][:80]}')
    print(f'    TEXT : {str(row["text"])[:120]}')
    print()

## 2. The clean_text() Pipeline

8 sequential operations:

1. **Strip URLs** — remove `http://`, `www.` links
2. **Strip HTML tags** — remove `<b>`, `<a href>`, etc.
3. **Remove handles/hashtags** — strip `@user`, `#symbol`
4. **Remove special chars and digits** — keep only letters
5. **TweetTokenizer** — preserves contractions, emoticon-aware
6. **Filter stopwords** — NLTK English + domain stops (Reuters, AP, weekday names)
7. **Filter short tokens** — drop tokens under 3 characters
8. **Lemmatize** — `attacking` → `attack`, `sanctions` → `sanction`

In [ ]:
from src.preprocessing import clean_text

samples = raw_df.sample(5, random_state=42)
for _, row in samples.iterrows():
    original = str(row['title']) + ' ' + str(row['text'])
    cleaned  = clean_text(original)
    print(f'BEFORE: {original[:100]}')
    print(f'AFTER : {cleaned[:100]}')
    print()

## 3. Full Preprocessing Pipeline

In [ ]:
from src.preprocessing import preprocess_news

proc_df = preprocess_news(raw_df, save=True)
print(f'Processed shape: {proc_df.shape}')
proc_df[['title', 'cleaned_text', 'token_count', 'date']].head()

## 4. Token Count Analysis

In [ ]:
print('Token count statistics:')
print(proc_df['token_count'].describe().round(2))
print(f'\nArticles with < 5 tokens : {(proc_df["token_count"] < 5).sum()}')
print(f'Articles with 5-20 tokens: {((proc_df["token_count"] >= 5) & (proc_df["token_count"] <= 20)).sum()}')
print(f'Articles with > 20 tokens: {(proc_df["token_count"] > 20).sum()}')

## 5. Vocabulary Statistics

In [ ]:
all_tokens = proc_df['cleaned_text'].str.split().explode()
print(f'Total tokens (with repeats): {len(all_tokens)}')
print(f'Unique vocabulary         : {all_tokens.nunique()}')
print('\nTop 25 most frequent tokens:')
print(all_tokens.value_counts().head(25).to_string())

## Key Findings

| Metric | Value |
|---|---|
| Articles after preprocessing | 314 |
| Articles removed | 0 |
| Unique vocabulary | ~930 tokens |
| Top tokens | iran, oil, israel, sanction, war |

> **Next step:** `03_EDA_Sentiment_Overview.ipynb`